In [1]:
!uv tree

learning-mcp v0.1.0
├── chromadb v1.1.1
│   ├── bcrypt v5.0.0
│   ├── build v1.3.0
│   │   ├── colorama v0.4.6
│   │   ├── packaging v25.0
│   │   └── pyproject-hooks v1.2.0
│   ├── grpcio v1.75.1
│   │   └── typing-extensions v4.15.0
│   ├── httpx v0.28.1
│   │   ├── anyio v4.11.0
│   │   │   ├── idna v3.11
│   │   │   ├── sniffio v1.3.1
│   │   │   └── typing-extensions v4.15.0
│   │   ├── certifi v2025.10.5
│   │   ├── httpcore v1.0.9
│   │   │   ├── certifi v2025.10.5
│   │   │   └── h11 v0.16.0
│   │   └── idna v3.11
│   ├── importlib-resources v6.5.2
│   ├── jsonschema v4.25.1
│   │   ├── attrs v25.4.0
│   │   ├── jsonschema-specifications v2025.9.1
│   │   │   └── referencing v0.36.2
│   │   │       ├── attrs v25.4.0
│   │   │       ├── rpds-py v0.27.1
│   │   │       └── typing-extensions v4.15.0
│   │   ├── referencing v0.36.2 (*)
│   │   ├── rpds-py v0.27.1
│   │   ├── fqdn v1.5.1 (extra: format-nongpl)
│   │   ├── idna v3.11 (extra: format-nongpl)
│   │   ├── isoduration v20

Resolved 221 packages in 5ms


# 0. Imports and Configuration Setup

In [2]:
# ----- IMPORT -----
import os
import shutil # to remove database
import hashlib # create content's hash
import json
from pathlib import Path
from typing import List, Set, Dict, Any


# ----- LANGCHAIN IMPORTS -----
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document


# ----- FASTMCP IMPORT -----
# from fastmcp import FastMCP # direct import
from mcp.server.fastmcp import FastMCP

In [3]:
# ----- CONSTANTS -----
# current_dir = Path(__file__).parent.absolute() # not work in notebook
current_dir = Path.cwd()
current_dir

WindowsPath('c:/Users/user/Desktop/learning_mcp')

In [4]:
# ----- CONFIGURATION -----
CHROMA_DB_ROOT = os.path.join(current_dir, "research_chroma_dbs")
OLLAMA_BASE_URL = "http://localhost:11434"
EMBED_MODEL = "nomic-embed-text"

# 1. Initialization

In [5]:
# ----- MCP SERVER INITIALIZATION -----
mcp = FastMCP("Research Assistant")


In [6]:
# ----- OLLAMA EMBEDDING INITIALIZATION -----
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL
)

# 2. Utility Functions

We'll implement the following utility functions:

1. `def get_content_hash(content: str) -> str:` to get hash value of the content
2. `def load_content_hashes(topic_path: Path) -> Set[str]:` to load the (already hashed) stored content
3. `def save_content_hashes(topic_path: Path, hashes: Set[str]):` to store hashes
4. `def get_vectorstore(topic: str) -> Chroma:` to create vectorstore for a topic

### Experiment

In [7]:
text = "hello, 你好"
encoded_text = text.encode("utf-8")
encoded_text

b'hello, \xe4\xbd\xa0\xe5\xa5\xbd'

In [8]:
hashlib.md5(encoded_text)

<md5 _hashlib.HASH object @ 0x000002B413EDE970>

In [9]:
hashed_text = hashlib.md5(encoded_text).hexdigest()
print(f"hash length: {len(hashed_text)}")
hashed_text

hash length: 32


'67cb377af7afba7805f18f056feae8a0'

### 2.1 `get_content_hash()`

In [10]:
def get_content_hash(content: str) -> str:
    """Generate a hash for content to check for duplication."""
    return hashlib.md5(content.encode("utf-8")).hexdigest()

### 2.2 `load_content_hashes()`

In [11]:
def load_content_hashes(topic_path: Path) -> Set[str]:
    """Load existing content hashes from metadata file."""
    metadata_file = topic_path / "content_hashes.json"
    if metadata_file.exists():
        try:
            with open(metadata_file, "r") as f:
                return set(json.load(f))
        except:
            return set()
    return set()

### 2.3 `save_content_hashes()`

In [12]:
def save_content_hashes(topic_path: Path, hashes: Set[str]): # hashes created from get_content_hash()
    """Save content hashes to metadata file."""
    metadata_file = topic_path / "content_hashes.json"
    with open(metadata_file, 'w') as f:
        json.dump(list(hashes), f)

### 2.4 `get_vectorstore()`

In [13]:
def get_vectorstore(topic: str) -> Chroma:
    """Get or create a ChromaDB vectorstore for a topic."""
    topic_path = CHROMA_DB_ROOT / topic
    topic_path.mkdir(parents=True, exist_ok=True)

    return Chroma(
        persist_directory=str(topic_path),
        embedding_function=embeddings,
        collection_name=f"research_on_{topic}"
    )

# 3. MCP Tools Implementations

In this section, we'll implement the following tools:

1. `def save_research_data(content: List[str], topic: str = "default") -> str:` to save (multiple) content of a given topic to Chroma db and return successful message
2. `def search_research_data(query: str, topic: str = "default", max_results: int = 5) -> str:` to search from db with the given query and topic
3. `def list_research_topics() -> str:` to list all topics in db
4. `def delete_research_topic(topic: str) -> str:` to delete certain topic
5. `def get_topic_info(topic: str) -> str:` to get metadata/info of given topic

### 3.1 `save_research_data()`

In [14]:
@mcp.tool()
def save_research_data(content: List[str], topic: str = "default") -> str:
    """
    Save research content to vector database for future retrieval.
    Args:
        content: List of text content to save
        topic: Topic name for organizing the data (creates separate DB)
    """
    try:
        topic_path = CHROMA_DB_ROOT / topic
        topic_path.mkdir(parents=True, exist_ok=True)

        # Load existing content hashes
        existing_hashes = load_content_hashes(topic_path)

        # Filter out duplicate content
        new_content = []
        new_hashes = set(existing_hashes)

        for text in content:
            content_hash = get_content_hash(text)
            if content_hash not in existing_hashes:
                new_content.append(text)
                new_hashes.add(content_hash)

        if not new_content:
            return f"No new content so save = all {len(content)} documents already exist in topic: {topic}"

        # Get vectorstore for this topic
        vectorstore = get_vectorstore(topic)

        # Create documents with metadata
        documents = []
        doc_ids = []
        for i, text in enumerate(new_content):
            content_hash = get_content_hash(text)
            doc = Document(
                page_content=text,
                metadata={
                    "topic": topic,
                    "content_hash": content_hash,
                    "doc_index": len(existing_hashes) + i
                }
            )
            documents.append(doc)
            doc_ids.append(f"{topic}_{content_hash}")

        # Add documents to vectorstore
        vectorstore.add_documents(documents=documents, ids=doc_ids)

        # Save updated content hashes
        save_content_hashes(topic_path, new_hashes)

        return f"Successfully saved {len(new_content)} new documents to topic: {topic} (skipped {len(content) - len(new_content)} duplicates)"

    except Exception as e:
        return f"Error saving research data: {str(e)}"


### 3.2 `search_research_data()`

In [15]:
@mcp.tool()
def search_research_data(query: str, topic: str = "default", max_results: int = 5) -> str:
    """
    Search through saved research data using semantic similarity.
    Args:
        query: Search query
        topic: Topic database to search in
        max_results: Maximum number of results to return
    """
    try:
        topic_path = CHROMA_DB_ROOT / topic

        if not topic_path.exists():
            return f"No research data found for topic: {topic}"

        # Create/Get vectorstore for this topic
        vectorstore = get_vectorstore(topic)

        # Check if collection has any documents
        try:
            collection = vectorstore._collection
            count = collection.count()
            if count == 0:
                return f"No documents found in topic: {topic}"
        except:
            return f"No research data found for topic: {topic}"

        # Search for similar documents
        results = vectorstore.similarity_search_with_score(query, k=max_results)

        if not results:
            return f"No relevant results found for query: '{query}' in topic: {topic}"

        # Format results
        formatted_results = []
        for i, (doc, score) in enumerate(results):
            similarity = 1 - score # Convert distance to similarity
            result_text = f"Result {i+1} (Similarity: {similarity:.3f}):\n {doc.page_content}\n"
            formatted_results.append(result_text)

        return "\n" + "="*50 + "\n".join(formatted_results) + "="*50

    except Exception as e:
        return f"Error searching research data: {str(e)}"

### 3.3 `list_research_topics()`

In [16]:
@mcp.tool()
def list_research_topics() -> str:
    """
    List all available research topics (vector databases).
    """
    try:
        if not CHROMA_DB_ROOT.exists():
            return "No research topics found"
        
        topics = []
        for path in CHROMA_DB_ROOT.iterdir():
            if path.is_dir():
                # Try to get document count from ChromaDB
                try:
                    vectorstore = get_vectorstore(path.name)
                    collection = vectorstore._collection
                    doc_count = collection.count()
                    topics.append(f"Topic: {path.name} ({doc_count} documents)")
                except Exception as e:
                    # Fallback to hash count if ChromaDB fails
                    try:
                        hashes = load_content_hashes(path)
                        doc_count = len(hashes)
                        topics.append(f"Topic: {path.name} ({doc_count} documents)")
                    except:
                        topics.append(f"Topic: {path.name}")

        if not topics:
            return "No research topics found"

        return "\n".join(topics)

    except Exception as e:
        return f"Error listing topics: {str(e)}"


### 3.4 `delete_research_topic()`

In [17]:
@mcp.tool()
def delete_research_topic(topic: str) -> str:
    """
    Delete a research topic and all its data.
    Args:
        topic: Topic name to delete
    """
    try:
        topic_path = CHROMA_DB_ROOT / topic

        if not topic_path.exists():
            return f"Topic '{topic}' does not exist"

        # Try to delete the ChromaDB collection first
        try:
            vectorstore = get_vectorstore(topic)
            vectorstore.delete_collection()
        except:
            pass # Continue even if collection deletion fails

        # Remote the entire directory
        shutil.rmtree(topic_path)

        return f"Successfully deleted topic: {topic}"

    except Exception as e:
        return f"Error deleting topic: {str(e)}"


### 3.5 `get_topic_info()`

In [ ]:
@mcp.tool()
def get_topic_info(topic: str) -> str:
    """
    Get detailed information about a research topic.
    Args:
        topic: Topic name to get info for
    """
    try:
        topic_path = CHROMA_DB_ROOT / topic

        if not topic_path.exists():
            return f"Topic '{topic}'does not exist"

        # Get vectorstore info
        vectorstore = get_vectorstore(topic)
        collection = vectorstore._collection
        doc_count = collection.count()

        # Get content hashes info
        hashes = load_content_hashes(topic_path)
        hash_count = len(hashes)

        info = f"""Topic Information: {topic}
                - ChromaDB Collection: research_on_{topic}
                - Document Count: {doc_count}
                - Hash Records: {hash_count}
                - Database Path: {topic_path}
                - Embedding Model: {EMBED_MODEL}
                - Ollama URL: {OLLAMA_BASE_URL}"""

        return info

    except Exception as e:
        return f"Error getting topic info: {str(e)}"

# 4. MCP Client Setup